In [1]:
import torch, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from tabpfn_extensions import TabPFNClassifier           # 记得是 extensions 里的
from models.tabular_encoder import tabular_encoder_classifier

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
csv_path = rf"C:\Users\dongzj\Desktop\Multimodal_AD\adni_dataset\ADNI_Tabel.csv"
label_col  = "GROUP"
classes    = ["SMCI", "PMCI"]
# feature_cols = ["MMSE"]
n_fold     = 5
dropna     = False
test_size  = 0.2
start_col  = 4 

▶ Using device: cuda:0


In [146]:
# ---------------- 生成嵌入 -----------------
train_df, test_df = tabular_encoder_classifier(
    csv_path   = csv_path,
    label_col  = label_col,
    classes    = classes,
    # feature_cols = feature_cols,
    n_fold     = n_fold,
    dropna     = dropna,
    test_size  = test_size,
    start_col  = start_col
)

# ---------- 拆分 X / y ----------
X_train = train_df.drop(columns=["label"]).values       # shape (N_tr, 192)
y_train = train_df["label"].values.astype("int64")

X_test  = test_df.drop(columns=["label"]).values
y_test  = test_df["label"].values.astype("int64")

print("Train:", X_train.shape, "Test:", X_test.shape)

✓ train_emb (338, 192) · test_emb (85, 192)
已保存到: train_embeddings.csv  /  test_embeddings.csv
Train: (338, 192) Test: (85, 192)


In [147]:
# ---------- 3. 用 TabPFNClassifier 做最终分类 ----------
clf = TabPFNClassifier(device=DEVICE)
clf.fit(X_train, y_train)

y_prob = clf.predict_proba(X_test)[:, 1]                # P(Positive)
y_pred = (y_prob >= 0.5).astype("int64")                # 默认 0.5 阈值

# ---------- 4. 评估 ----------
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nAccuracy = {acc:.4f} · AUC = {auc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=classes))



Accuracy = 1.0000 · AUC = 1.0000

              precision    recall  f1-score   support

          CN       1.00      1.00      1.00        41
          AD       1.00      1.00      1.00        44

    accuracy                           1.00        85
   macro avg       1.00      1.00      1.00        85
weighted avg       1.00      1.00      1.00        85



In [148]:
def quick_eval_from_saved(train_csv="train_embeddings.csv", test_csv="test_embeddings.csv"):
    """
    读取带标签的嵌入 CSV，使用 **最简 SVM** (线性核) 做一次快速评估。
    """
    from sklearn.svm import SVC
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline

    tr = pd.read_csv(train_csv)
    te = pd.read_csv(test_csv)

    y_tr, X_tr = tr["label"].values, tr.drop(columns="label").values
    y_te, X_te = te["label"].values, te.drop(columns="label").values

    # 使用线性核 SVM, 外加标准化, 这是最简易且常用的组合
    clf = make_pipeline(StandardScaler(), SVC(kernel="linear"))
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    print(f"[quick eval · SVM-linear] Accuracy on {test_csv}: {acc:.4f}")
    return acc

quick_eval_from_saved()

[quick eval · SVM-linear] Accuracy on test_embeddings.csv: 1.0000


1.0

In [149]:
from sklearn.svm import SVC
import shap
import pandas as pd

# 1. 读 CSV
train_df = pd.read_csv("train_embeddings.csv")
X_tr = train_df.drop(columns="label").values
y_tr = train_df["label"].values

# 2. 训练 SVM
svm = SVC(kernel="linear", probability=True).fit(X_tr, y_tr)

# 3. KernelExplainer
background = shap.sample(X_tr, 100, random_state=42)
explainer = shap.KernelExplainer(svm.predict_proba, background)

# 4. 计算 SHAP
test_df = pd.read_csv("test_embeddings.csv")
X_te = test_df.drop(columns="label").values

# shap_vals 是一个 list，[neg_class, pos_class]
shap_vals = explainer.shap_values(X_te, nsamples=200)

# 我们只关心正类 (索引 1)：
sv = shap_vals[1]  # shape = (n_samples, n_features + 1)

# **关键**：去掉最后一列常数偏置项
sv_trim = sv[:, :-1]  # 现在 shape = (n_samples, n_features)

# sanity check
assert sv_trim.shape[1] == X_te.shape[1], (
    f"SHAP 列数({sv_trim.shape[1]})应等于特征数({X_te.shape[1]})"
)

# 5. 可视化
feature_names = list(train_df.columns[1:])  # 嵌入特征的列名

# — 全局特征重要性柱状图 —
shap.summary_plot(
    shap_values=sv_trim,
    features=X_te,
    feature_names=feature_names,
    plot_type="bar"
)

# — 蜂群图 —
shap.summary_plot(
    shap_values=sv_trim,
    features=X_te,
    feature_names=feature_names
)


100%|██████████| 85/85 [00:14<00:00,  5.95it/s]


AssertionError: SHAP 列数(1)应等于特征数(192)